# LTP – Phase 1: Data Quality & EDA (Pink Label Case)


Notebook gerado automaticamente. Objetivo: executar limpeza de dados (Data Quality) e uma
primeira EDA para preparar o Dashboard (2nd Challenge).

**Ficheiros de origem**
- `/mnt/data/Data_store.xlsx`
- `/mnt/data/Data_labels.xlsx`

**Saídas**
- Limpo: `/mnt/data/ltp_clean_phase1.csv`
- Log: `/mnt/data/ltp_clean_phase1_log.csv`


In [1]:

import pandas as pd
import numpy as np
from datetime import datetime


In [2]:

def euro_to_float(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip()
    s = s.replace("%", "")
    allowed = set("0123456789-.,")
    s = "".join(ch for ch in s if ch in allowed)
    if s.count(",") > 0 and s.count(".") > 0:
        s = s.replace(".", "").replace(",", ".")
    elif s.count(",") > 0 and s.count(".") == 0:
        s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan

def parse_price_and_discount(val):
    if pd.isna(val):
        return np.nan, np.nan
    s = str(val)
    disc = np.nan
    if "(" in s and ")" in s:
        inside = s[s.find("(")+1:s.find(")")]
        try:
            disc = float(inside.replace("%","").replace(",",".")) / 100.0
        except:
            disc = np.nan
    price_part = s.split("(")[0].strip()
    price = euro_to_float(price_part)
    return price, disc

def safe_pct(x):
    try:
        return float(x)
    except:
        return euro_to_float(x)


## 1) Carregar dados

In [3]:

store_path = "Data_store.xlsx"
labels_path = "Data_labels.xlsx"

store = pd.ExcelFile(store_path).parse(0)
labels = pd.ExcelFile(labels_path).parse(0)

df = store.copy()
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
len(df), df.head(3)


(342,
    idstore   type  selling_square_ft  district
 0        1  Large             4762.0     Évora
 1        2  Large            12358.0  Bragança
 2        3  Large            16159.0    Lisboa)

## 2) Limpeza de `brand`

In [4]:

if 'brand' in df.columns:
    df['brand'] = df['brand'].astype(str).str.strip()
    df['brand'] = df['brand'].str.replace(r"\bmarca\b","Marca", regex=True)
    df['brand'] = df['brand'].str.replace("ca","ca ", regex=False)
df['brand'].head(10) if 'brand' in df.columns else "brand not found"


'brand not found'

## 3) `new_pvp` e `discount`

In [5]:

if 'new_pvp' in df.columns:
    parsed = df['new_pvp'].apply(parse_price_and_discount)
    df['new_pvp_parsed'] = parsed.apply(lambda t: t[0])
    df['discount_parsed'] = parsed.apply(lambda t: t[1])

    newpvp_num = df['new_pvp'].apply(euro_to_float)
    df['new_pvp'] = newpvp_num.fillna(df['new_pvp_parsed'])

    if 'discount' in df.columns:
        disc_num = df['discount'].apply(safe_pct)
        if disc_num.dropna().gt(1).mean() > 0.5:
            disc_num = disc_num/100.0
        df['discount'] = disc_num.fillna(df['discount_parsed'])
    else:
        df['discount'] = df['discount_parsed']

    df.drop(columns=['new_pvp_parsed','discount_parsed'], inplace=True, errors='ignore')

df[['new_pvp','discount']].head(10) if 'new_pvp' in df.columns else "new_pvp not found"


'new_pvp not found'

## 4) Imputação de `oldpvp`

In [6]:

if set(['oldpvp','new_pvp']).issubset(df.columns):
    old_num = df['oldpvp'].apply(euro_to_float)
    new_num = df['new_pvp'].apply(euro_to_float)
    disc = df['discount'] if 'discount' in df.columns else np.nan
    disc = disc.apply(lambda x: x/100.0 if (pd.notna(x) and x>1) else x)
    est_old = new_num / (1 - disc)
    df['oldpvp'] = old_num.fillna(est_old)
df[['oldpvp','new_pvp','discount']].head(10) if 'oldpvp' in df.columns else "oldpvp not found"


'oldpvp not found'

## 5) Remover `labelqty` (sem variância)

In [7]:

if 'labelqty' in df.columns and df['labelqty'].nunique(dropna=False) <= 1:
    df.drop(columns=['labelqty'], inplace=True)
list(df.columns)


['idstore', 'type', 'selling_square_ft', 'district']

## 6) `weight` – missing por SKU depois global

In [8]:

if 'weight' in df.columns:
    df['weight'] = df['weight'].apply(euro_to_float)
    if 'sku' in df.columns:
        df['weight'] = df.groupby('sku')['weight'].transform(lambda s: s.fillna(s.median()))
    df['weight'] = df['weight'].fillna(df['weight'].median())
df['weight'].describe() if 'weight' in df.columns else "weight not found"


'weight not found'

## 7) `margin` normalizada para fração (0–1)

In [9]:

if 'margin' in df.columns:
    m = df['margin'].apply(safe_pct)
    if m.dropna().gt(1).mean() > 0.5:
        m = m/100.0
    df['margin'] = m
df['margin'].describe() if 'margin' in df.columns else "margin not found"


'margin not found'

## 8) `profit` – validação e remoção de outliers vs `margin*new_pvp`

In [10]:

if set(['profit','new_pvp','margin']).issubset(df.columns):
    df['profit'] = df['profit'].apply(euro_to_float)
    expected = df['margin'] * df['new_pvp']
    diff = (df['profit'] - expected).abs()
    rel = diff / (expected.replace(0,np.nan).abs())
    bad = (diff > 1.0) & (rel > 0.10)
    df = df.loc[~bad].copy()
len(df)


342

## 9) Datas – parse e consistência

In [12]:
# --- Resolver nomes equivalentes antes de mexer nas datas ---
# (garante nomes canónicos: expiring_date, labelling_date, sell_date)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

aliases = {
    "expiring_date": ["expiring_date","expiry_date","expiration_date","exp_date",
                      "best_before","best_before_date","valid_to","validity_end"],
    "labelling_date": ["labelling_date","labeling_date","label_date","pink_label_date",
                       "labelling_dt","label_dt"],
    "sell_date": ["sell_date","sale_date","sold_date","selling_date",
                  "sales_date","sold_dt","sell_dt"]
}

def ensure_col(df, canonical, options, contains=None):
    # 1) tenta nomes exatos
    for opt in options:
        if opt in df.columns:
            df[canonical] = df[opt]
            return opt
    # 2) tenta fuzzy por tokens
    if contains:
        for c in df.columns:
            if any(tok in c for tok in contains):
                df[canonical] = df[c]
                return c
    # 3) se nada encontrado, cria coluna vazia adequada
    import numpy as np, pandas as pd
    df[canonical] = pd.NaT if "date" in canonical else np.nan
    return None

src_exp = ensure_col(df, "expiring_date", aliases["expiring_date"], contains=["expir","best","bbd"])
src_lab = ensure_col(df, "labelling_date", aliases["labelling_date"], contains=["label"])
src_sel = ensure_col(df, "sell_date",      aliases["sell_date"],      contains=["sell","sale","sold"])
print("Resolved -> expiring:", src_exp, "| labelling:", src_lab, "| sell:", src_sel)

# --- Parse de datas e regras de consistência ---
import pandas as pd
def parse_date(series):
    if series.dtype == "datetime64[ns]":
        return series
    return pd.to_datetime(series.astype(str).str.replace("-", "/"),
                          errors="coerce", dayfirst=True, infer_datetime_format=True)

for col in ["expiring_date", "labelling_date", "sell_date"]:
    if col in df.columns:
        df[col] = parse_date(df[col])

# Remover vendas antes da etiquetagem (se ambas existirem)
if {"sell_date","labelling_date"}.issubset(df.columns):
    invalid = df["sell_date"].notna() & df["labelling_date"].notna() & (df["sell_date"] < df["labelling_date"])
    df = df.loc[~invalid].copy()

# Mostrar só as colunas que existirem para evitar KeyError
cols_show = [c for c in ["expiring_date","labelling_date","sell_date"] if c in df.columns]
df[cols_show].head(10)


Resolved -> expiring: None | labelling: None | sell: selling_square_ft


C:\Users\rpinto\AppData\Local\Temp\ipykernel_12756\3790333575.py:41: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(series.astype(str).str.replace("-", "/"),
C:\Users\rpinto\AppData\Local\Temp\ipykernel_12756\3790333575.py:41: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(series.astype(str).str.replace("-", "/"),


,expiring_date,labelling_date,sell_date
0,NaT,NaT,NaT
1,NaT,NaT,NaT
2,NaT,NaT,NaT
3,NaT,NaT,NaT
4,NaT,NaT,NaT
5,NaT,NaT,NaT
6,NaT,NaT,NaT
7,NaT,NaT,NaT
8,NaT,NaT,NaT
9,NaT,NaT,NaT


## 10) Flag `sold` coerente com `sell_date`

In [13]:

if 'sell_date' in df.columns:
    sold_comp = df['sell_date'].notna().astype(int)
    if 'sold' in df.columns:
        df.loc[sold_comp==1,'sold'] = 1
        df['sold'] = df['sold'].fillna(0).astype(int)
    else:
        df['sold'] = sold_comp
df['sold'].value_counts(dropna=False) if 'sold' in df.columns else "sold not found"


sold
0    342
Name: count, dtype: int64

## 11) Variáveis derivadas (buckets e expiração no dia)

In [14]:

if set(['sell_date','expiring_date']).issubset(df.columns):
    df['sold_on_expiration'] = (df['sell_date'].dt.date == df['expiring_date'].dt.date).astype(int)

if 'discount' in df.columns:
    import pandas as pd
    bins = [-0.01, 0.10, 0.20, 0.30, 0.50, 1.00]
    labels = ["0–10%","10–20%","20–30%","30–50%","50%+"]
    try:
        df['discount_bucket'] = pd.cut(df['discount'].astype(float), bins=bins, labels=labels)
    except Exception:
        disc_fix = df['discount'].apply(safe_pct)
        if disc_fix.dropna().gt(1).mean() > 0.5:
            disc_fix = disc_fix/100.0
        df['discount_bucket'] = pd.cut(disc_fix, bins=bins, labels=labels)
df[['discount','discount_bucket']].head(10) if 'discount' in df.columns else "discount not found"


'discount not found'

## 12) Quick EDA (tabelas essenciais)

In [15]:

summary = {}
if 'sold' in df.columns:
    summary['sold_rate'] = df['sold'].mean()
if set(['sold','expiring_date']).issubset(df.columns):
    summary['sold_on_expiration_rate'] = df.loc[df['sold']==1, 'sold_on_expiration'].mean()

sold_by_store = df.groupby('idstore')['sold'].mean().sort_values(ascending=False) if 'idstore' in df.columns and 'sold' in df.columns else None
sold_by_brand = df.groupby('brand')['sold'].mean().sort_values(ascending=False) if 'brand' in df.columns and 'sold' in df.columns else None
sold_by_bucket = df.groupby('discount_bucket')['sold'].mean().sort_values(ascending=False) if 'discount_bucket' in df.columns and 'sold' in df.columns else None

summary, sold_by_store.head(10) if sold_by_store is not None else None


({'sold_rate': np.float64(0.0), 'sold_on_expiration_rate': nan},
 idstore
 342    0.0
 1      0.0
 2      0.0
 3      0.0
 4      0.0
 5      0.0
 326    0.0
 325    0.0
 324    0.0
 323    0.0
 Name: sold, dtype: float64)

## 13) Exportar limpo para Power BI

In [16]:

clean_path = "/mnt/data/ltp_clean_phase1.csv"
df.to_csv(clean_path, index=False)
clean_path


OSError: Cannot save file into a non-existent directory: '\mnt\data'